# 09 · Unannotated-record Recovery

**Purpose.** Validate the tool's *core reason to exist*: assigning annotation to records that have **no** (or non-useful) annotation. Units 03/04 prove *coordinate transfer is correct when the query already carries truth*; this notebook proves the tool can **recover that annotation when it is absent**.

Two complementary tests:
1. **Strip test** — take annotated records, delete their features, re-annotate via tblastn lifting, and score against the original annotation (which serves as truth).
2. **Real noAnno records** — run directly on `FMD_OQ211398_noAnno.gb` and `PRRS_PP946131_noAnno.gb` to demonstrate end-to-end annotation on genuinely unannotated input.

## Inputs

Annotated query sets (as truth source) + reference; plus the two real noAnno GenBank files.

In [ ]:
from pathlib import Path
from copy import deepcopy
import sys

import pandas as pd
import matplotlib.pyplot as plt

# --- anchor ROOT to the repo (folder that contains app/src) ---
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "app" / "src").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA = ROOT / "app" / "data"

FMD_REF    = DATA / "FMD"  / "FMD_ref_test.gb"
FMD_QUERY  = DATA / "FMD"  / "FMD_100seq_anno.gb"
PRRS_REF   = DATA / "PRRS" / "PRRS_ref_test.gb"
PRRS_QUERY = DATA / "PRRS" / "PRRS_100seq_anno.gb"
PED_REF    = DATA / "PED"  / "PED_ref_1.gb"
PED_QUERY  = DATA / "PED"  / "PED_100seqs.gb"

# real unannotated records already in the repo:
FMD_NOANNO  = DATA / "FMD"  / "FMD_OQ211398_noAnno.gb"
PRRS_NOANNO = DATA / "PRRS" / "PRRS_PP946131_noAnno.gb"

RUN_FULL = False
SAMPLE_N = 10


In [ ]:
UNIT_DIR = ROOT / "app" / "validation" / "09_recovery_unannotated"
OUT = UNIT_DIR / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
print("outputs ->", OUT)

## Setup — modules

In [ ]:
from app.src.io.genbank_parser import load_genbank_records, load_single_genbank, parse_cds_features
from app.src.alias.gene_alias import apply_alias_to_features
from app.src.features.annotation_strategy import get_strategy
from app.src.lifting.tblastn_lifter import lift_all_tblastn
from app.validation._shared.validation_utils import (
    load_reference_bundle, lifted_to_rows, compare_predictions_to_truth, summarize_comparison,
)

def strip_annotation(record):
    """Return a deep copy of the record with all features removed (sequence kept)."""
    bare = deepcopy(record)
    bare.features = []
    return bare

## Test 1 — Strip test (annotate → strip → re-lift → compare)

For each record we keep the original (alias-normalised) annotation as **truth**, hand the tool a de-annotated copy, and confirm (a) the router sends it to **tblastn** and (b) the recovered coordinates match truth.

In [ ]:
DATASETS = {"fmd": (FMD_REF, FMD_QUERY), "prrs": (PRRS_REF, PRRS_QUERY), "ped": (PED_REF, PED_QUERY)}

rows = []
for label, (ref_path, query_path) in DATASETS.items():
    bundle = load_reference_bundle(ref_path)
    records = load_genbank_records(query_path)
    records = records if RUN_FULL else records[:SAMPLE_N]
    for rec in records:
        # truth = original annotation, alias-normalised
        truth = apply_alias_to_features(parse_cds_features(rec), bundle["alias_lookup"])
        truth = [t for t in truth if t.get("name_source") in ("alias", "alias_conflict_resolved")]
        if not truth:
            continue
        bare = strip_annotation(rec)
        strategy, _ = get_strategy(bare, bundle["alias_lookup"])   # expect "tblastn"
        lifted = lift_all_tblastn(
            ref_features=bundle["features"], ref_record=bundle["record"], query_record=bare,
            validate_codons=(bundle["feature_type"] == "CDS"),
        )
        preds = lifted_to_rows(rec.id, lifted, "tblastn")
        compared = compare_predictions_to_truth(preds, truth)
        compared.insert(0, "virus", label)
        compared["routed_to"] = strategy
        rows.append(compared)

strip_df = pd.concat(rows, ignore_index=True)
strip_df.to_csv(OUT / "strip_recovery_per_prediction.tsv", sep="\t", index=False)
print("routing check (should be all tblastn):", strip_df["routed_to"].value_counts().to_dict())
strip_df.head()

## Metrics — recovery accuracy on de-annotated records

In [ ]:
recovery = summarize_comparison(strip_df, ["virus"])
recovery.to_csv(OUT / "strip_recovery_accuracy.tsv", sep="\t", index=False)
recovery

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(recovery["virus"], recovery["exact_pct"], label="exact")
ax.bar(recovery["virus"], recovery["coord_pct"] - recovery["exact_pct"],
       bottom=recovery["exact_pct"], label="coord-only")
ax.set_ylabel("%"); ax.set_ylim(0, 100); ax.legend()
ax.set_title("Annotation recovery on de-annotated records")
fig.tight_layout(); fig.savefig(OUT / "strip_recovery_accuracy.png", dpi=200)

## Test 2 — Real unannotated GenBank records

Run the tool on the two genuinely unannotated files. There is no per-record truth here, so this is a **demonstration**: confirm routing = tblastn and that the tool emits a plausible gene table (names, coordinates, coverage, identity).

In [ ]:
noanno = {"fmd": (FMD_REF, FMD_NOANNO), "prrs": (PRRS_REF, PRRS_NOANNO)}
demo_rows = []
for label, (ref_path, q_path) in noanno.items():
    bundle = load_reference_bundle(ref_path)
    rec = load_single_genbank(q_path)
    strategy, _ = get_strategy(rec, bundle["alias_lookup"])
    print(f"{label}: {rec.id} routed_to={strategy}")
    lifted = lift_all_tblastn(
        ref_features=bundle["features"], ref_record=bundle["record"], query_record=rec,
        validate_codons=(bundle["feature_type"] == "CDS"),
    )
    for r in lifted_to_rows(rec.id, lifted, strategy):
        r["virus"] = label; demo_rows.append(r)

demo_df = pd.DataFrame(demo_rows)
demo_df.to_csv(OUT / "noanno_demo_predictions.tsv", sep="\t", index=False)
demo_df[["virus", "record_id", "pred_name", "pred_start", "pred_end", "strand", "coverage", "identity", "status"]]

## Interpretation

> ⚠️ **TODO**: 1–2 câu: tool phục hồi được annotation ở record trắng (exact/coord ~X%), và tạo được bảng gene hợp lý cho 2 record noAnno thật. Đây là bằng chứng trực tiếp cho use case chính (approach 2 trong brief).